In [1]:
from simulation import citygraph_dataset
# from learning import inductive_route_learning, eval_route_generator, bee_colony
from learning.bee_colony import main as main_bee  # я так обозвал
from learning.eval_route_generator import main as main_eval # я так обозвал
from learning.nsgaii import main as main_ngsaii
# from learning.simulated_annealing import main as main_sa
# from learning.bagloee import main as main_bagloo
from omegaconf import OmegaConf, DictConfig
from simulation import drawing

from tqdm import tqdm
from pathlib import Path

In [2]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
import os

cfg_dir = os.path.abspath("../TNDP_learning/cfg")

In [4]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from learning.eval_route_generator import main as main_eval
from learning.bee_colony import main as main_bee
from learning.nsgaii import main as main_nsgaii  # Adjust the import based on the actual module
import torch
from tqdm import tqdm

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

# Все параметры экспериментов
experiments = [
    ("mumford0", 0.5, 0.5, 0, "origin"),
    ("mumford0", 0, 0, 1, "ours"),
]

# Путь к весам модели
model_weights_path = str(Path("../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt").resolve())

# CSV файл для результатов
results_file = Path("experiment_results.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "dataset", "param_set", "demand_time", "route_time", "connectivity", "algorithm",
            "ATT", "RTT", "median_connectivity","median_connectivity_weighted", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов с tqdm
for dataset_name, dt, rt, ct, param_name in tqdm(experiments, desc="Running experiments"):
    try:
        experiment_name = f"{dataset_name}_{param_name}"
        initial_routes_name = f"LC_{dataset_name}_{param_name}_starting"

        # LC (eval_model_mumford)
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_lc = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={initial_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}"
                ]
            )
        lc_metrics, _ = main_eval(cfg_lc)

        keys_order = ['ATT', 'RTT', 'median_connectivity', "median_connectivity_weighted", 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [dataset_name.lower(), param_name, dt, rt, ct, "LC"] + [round(lc_metrics[k].item(), 4) if k in lc_metrics else 0 for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

        # NeuroBCO (neural_bco_mumford)
        neuro_generated_name = f"NeuroBCO_{dataset_name}_{param_name}_generated"
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_neuro = compose(
                config_name="neural_bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={neuro_generated_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    f"init.path=output_routes/nn_construction_{initial_routes_name}_routes.pkl",
                ]
            )
        neuro_metrics, _ = main_bee(cfg_neuro)
        neuro_metrics['median_connectivity'] /= 60

        row = [dataset_name.lower(), param_name, dt, rt, ct, "NeuroBCO"] + [round(neuro_metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

        # BCO (bco_mumford)
        bco_generated_name = f"BCO_{dataset_name}_{param_name}_generated"
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_bco = compose(
                config_name="bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"++run_name={bco_generated_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    f"init.path=output_routes/nn_construction_{initial_routes_name}_routes.pkl",
                ]
            )
        bco_metrics, _ = main_bee(cfg_bco)
        bco_metrics['median_connectivity'] /= 60

        row = [dataset_name.lower(), param_name, dt, rt, ct, "BCO"] + [round(bco_metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

        # NSGAII
        nsgaii_generated_name = f"NSGAII_{dataset_name}_{param_name}_generated"
        use_weighted = param_name == "ours"
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_nsgaii = compose(
                config_name="nsgaii",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model_1.weights={model_weights_path}",
                    f"+model_2.weights={model_weights_path}",
                    f"++run_name={nsgaii_generated_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight=0.33",
                    f"++experiment.cost_function.kwargs.route_time_weight=0.33",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight=0.33",
                    f"++experiment.cost_function.kwargs.use_weighted_connectivity={'True' if use_weighted else 'False'}"
                ]
            )
        pareto_networks, pareto_costs, pareto_metrics = main_nsgaii(cfg_nsgaii)

        scores = dt * pareto_metrics['ATT'] + rt * pareto_metrics['RTT'] + ct * pareto_metrics['median_connectivity_weighted']
        idx = torch.argmin(scores).item()

        row = [dataset_name.lower(), param_name, dt, rt, ct, "NSGAII"] + [round(pareto_metrics[k][idx].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

    except Exception as e:
        print(f"[✗] Failed {experiment_name}: {e}")

Running experiments:  50%|█████     | 1/2 [00:45<00:45, 45.24s/it]

mutator use counts:
add_terminal: 31.0
delete_terminal: 11.0
add_inside: 6.0
delete_inside: 16.0
invert_nodes: 54.0
exchange_routes: 22.0
replace_node: 15.0
donate_node: 18.0
cost_based_grow: 35.0
cost_based_trim: 28.0
rpc_model_stochastic: 9.0
model_1_stochastic: 15.0
model_1_greedy: 25.0
model_2_stochastic: 35.0
model_2_greedy: 29.0


Running experiments: 100%|██████████| 2/2 [01:36<00:00, 48.10s/it]

mutator use counts:
add_terminal: 23.0
delete_terminal: 15.0
add_inside: 7.0
delete_inside: 33.0
invert_nodes: 45.0
exchange_routes: 23.0
replace_node: 18.0
donate_node: 13.0
cost_based_grow: 16.0
cost_based_trim: 24.0
rpc_model_stochastic: 21.0
model_1_stochastic: 19.0
model_1_greedy: 29.0
model_2_stochastic: 27.0
model_2_greedy: 36.0
